<a href="https://colab.research.google.com/github/Subuktageen-Farooqi/ms_course_deeplearning/blob/main/ms_deeplearning_tutorial_08a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Object Detection: Faster RCNN Pre-Trained - Tensorflow Implementation

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import cv2
import matplotlib.pyplot as plt
import random

# Step 1: Define the Backbone Network (Simple CNN)
def build_backbone(input_shape=(224, 224, 3)):
    inputs = keras.Input(shape=input_shape)
    x = keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    x = keras.layers.MaxPooling2D((2, 2))(x)
    x = keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling2D((2, 2))(x)
    x = keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling2D((2, 2))(x)
    outputs = keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    return keras.Model(inputs, outputs)

backbone = build_backbone()

# Step 2: Build the Region Proposal Network (RPN)
def build_rpn(feature_map):
    rpn_conv = keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu")(feature_map)
    rpn_class = keras.layers.Conv2D(9 * 2, (1, 1), activation="sigmoid")(rpn_conv)  # 9 anchors, 2 classes (object/non-object)
    rpn_bbox = keras.layers.Conv2D(9 * 4, (1, 1))(rpn_conv)  # 9 anchors, 4 bounding box values
    return rpn_class, rpn_bbox

# Step 3: Implement the ROI Pooling Layer
class ROIPoolingLayer(tf.keras.layers.Layer):
    def __init__(self, pool_size, **kwargs):
        super(ROIPoolingLayer, self).__init__(**kwargs)
        self.pool_size = pool_size

    def call(self, inputs):
        feature_map, rois = inputs
        rois = tf.reshape(rois, (-1, 4))
        batch_indices = tf.zeros((tf.shape(rois)[0],), dtype=tf.int32)
        pooled_rois = tf.image.crop_and_resize(
            feature_map, boxes=rois, box_indices=batch_indices, crop_size=self.pool_size
        )
        return pooled_rois

    def compute_output_shape(self, input_shape):
        feature_map_shape, rois_shape = input_shape
        num_rois = rois_shape[0]
        return (num_rois, self.pool_size[0], self.pool_size[1], feature_map_shape[-1])

roi_pooling_layer = ROIPoolingLayer((7, 7))

# Step 4: Create Classification and Regression Heads
def build_heads(pooled_rois, num_classes):
    flatten = keras.layers.Flatten()(pooled_rois)
    dense = keras.layers.Dense(1024, activation="relu")(flatten)
    classifier = keras.layers.Dense(num_classes, activation="softmax")(dense)
    bbox_regressor = keras.layers.Dense(num_classes * 4)(dense)  # 4 values per bounding box
    return classifier, bbox_regressor

# Step 5: Assemble the Complete Faster R-CNN Model
def build_faster_rcnn(num_classes, input_shape=(224, 224, 3)):
    input_image = keras.Input(shape=input_shape)
    rois = keras.Input(shape=(None, 4))  # ROIs input

    # Backbone (feature extraction)
    backbone = build_backbone(input_shape)
    feature_map = backbone(input_image)

    # RPN
    rpn_class, rpn_bbox = build_rpn(feature_map)

    # ROI Pooling
    pooled_rois = roi_pooling_layer([feature_map, rois])

    # Classification & Regression heads
    classifier, bbox_regressor = build_heads(pooled_rois, num_classes)

    # Build the complete model
    model = keras.Model(inputs=[input_image, rois], outputs=[rpn_class, rpn_bbox, classifier, bbox_regressor])
    return model

# Instantiate the model
num_classes = 80  # Example number of classes for COCO dataset
faster_rcnn = build_faster_rcnn(num_classes)
faster_rcnn.summary()

# Step 6: Calculate Intersection over Union (IoU)
def calculate_iou(box1, box2):
    ymin1, xmin1, ymax1, xmax1 = box1
    ymin2, xmin2, ymax2, xmax2 = box2

    inter_xmin = max(xmin1, xmin2)
    inter_ymin = max(ymin1, ymin2)
    inter_xmax = min(xmax1, xmax2)
    inter_ymax = min(ymax1, ymax2)

    inter_area = max(0, inter_xmax - inter_xmin) * max(0, inter_ymax - inter_ymin)
    area1 = (xmax1 - xmin1) * (ymax1 - ymin1)
    area2 = (xmax2 - xmin2) * (ymax2 - ymin2)
    union_area = area1 + area2 - inter_area

    iou = inter_area / union_area if union_area != 0 else 0
    return iou

# Step 7: Calculate Precision and Recall
def calculate_precision_recall(pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels, iou_threshold=0.5):
    tp, fp = 0, 0
    matched_gt = set()

    for i, pred_box in enumerate(pred_boxes):
        if pred_scores[i] < iou_threshold:
            continue

        best_iou = 0
        best_gt_index = -1

        for j, gt_box in enumerate(gt_boxes):
            if pred_labels[i] == gt_labels[j] and j not in matched_gt:
                iou = calculate_iou(pred_box, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_index = j

        if best_iou >= iou_threshold:
            tp += 1
            matched_gt.add(best_gt_index)
        else:
            fp += 1

    fn = len(gt_boxes) - len(matched_gt)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    return precision, recall

# Step 8: Calculate Average Precision (AP)
def calculate_ap(precisions, recalls):
    precisions = [0.0] + precisions + [0.0]
    recalls = [0.0] + recalls + [1.0]

    for i in range(len(precisions) - 1, 0, -1):
        precisions[i - 1] = max(precisions[i - 1], precisions[i])

    indices = [i for i in range(1, len(recalls)) if recalls[i] != recalls[i - 1]]
    ap = sum((recalls[i] - recalls[i - 1]) * precisions[i] for i in indices)
    return ap

# Step 9: Calculate Mean Average Precision (mAP) Across All Classes
def calculate_map(predictions, ground_truths, num_classes, iou_threshold=0.5):
    average_precisions = []

    for class_id in range(1, num_classes):  # Assuming class_id 0 is background
        all_precisions = []
        all_recalls = []

        for pred, gt in zip(predictions, ground_truths):
            pred_boxes = [box for i, box in enumerate(pred['boxes']) if pred['labels'][i] == class_id]
            pred_scores = [score for i, score in enumerate(pred['scores']) if pred['labels'][i] == class_id]
            gt_boxes = [box for i, box in enumerate(gt['boxes']) if gt['labels'][i] == class_id]

            precision, recall = calculate_precision_recall(
                pred_boxes, pred_scores, [class_id] * len(pred_boxes), gt_boxes, [class_id] * len(gt_boxes), iou_threshold
            )

            all_precisions.append(precision)
            all_recalls.append(recall)

        ap = calculate_ap(all_precisions, all_recalls)
        average_precisions.append(ap)

    mAP = np.mean(average_precisions)
    return mAP

# Placeholder predictions and ground truths for testing
predictions = [
    {'boxes': [[0.1, 0.1, 0.4, 0.4], [0.5, 0.5, 0.8, 0.8]], 'scores': [0.9, 0.7], 'labels': [1, 2]}
]
ground_truths = [
    {'boxes': [[0.12, 0.12, 0.42, 0.42], [0.5, 0.5, 0.8, 0.8]], 'labels': [1, 2]}
]

# Number of classes in the dataset (e.g., 80 for COCO)
num_classes = 80

# Calculate mAP
mean_ap = calculate_map(predictions, ground_truths, num_classes)
print(f"Mean Average Precision (mAP): {mean_ap:.2f}")

# PyTorch Implementation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.ops as ops
import numpy as np


# Step 1: Define the Backbone Network (Simple CNN)
class Backbone(nn.Module):
    def __init__(self, input_channels=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.features(x)


# Step 2: Build the Region Proposal Network (RPN)
class RPN(nn.Module):
    def __init__(self, in_channels=512, num_anchors=9):
        super().__init__()
        self.rpn_conv = nn.Conv2d(in_channels, 512, kernel_size=3, padding=1)
        self.rpn_class = nn.Conv2d(512, num_anchors * 2, kernel_size=1)
        self.rpn_bbox = nn.Conv2d(512, num_anchors * 4, kernel_size=1)

    def forward(self, feature_map):
        x = F.relu(self.rpn_conv(feature_map))
        rpn_class = torch.sigmoid(self.rpn_class(x))
        rpn_bbox = self.rpn_bbox(x)
        return rpn_class, rpn_bbox


# Step 3: Implement the ROI Pooling Layer
class ROIPoolingLayer(nn.Module):
    def __init__(self, pool_size=(7, 7), spatial_scale=1.0):
        super().__init__()
        self.pool_size = pool_size
        self.spatial_scale = spatial_scale

    def forward(self, feature_map, rois):
        # rois format: [N, 5] => [batch_idx, x1, y1, x2, y2]
        return ops.roi_align(
            feature_map,
            rois,
            output_size=self.pool_size,
            spatial_scale=self.spatial_scale,
            aligned=True,
        )


# Step 4: Create Classification and Regression Heads
class Heads(nn.Module):
    def __init__(self, in_channels=512, pool_size=(7, 7), num_classes=80):
        super().__init__()
        flattened = in_channels * pool_size[0] * pool_size[1]
        self.fc = nn.Linear(flattened, 1024)
        self.classifier = nn.Linear(1024, num_classes)
        self.bbox_regressor = nn.Linear(1024, num_classes * 4)

    def forward(self, pooled_rois):
        x = pooled_rois.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        class_logits = self.classifier(x)
        bbox_reg = self.bbox_regressor(x)
        class_probs = F.softmax(class_logits, dim=1)
        return class_probs, bbox_reg


# Step 5: Assemble the Complete Faster R-CNN Model
class FasterRCNNToy(nn.Module):
    def __init__(self, num_classes=80, pool_size=(7, 7)):
        super().__init__()
        self.backbone = Backbone()
        self.rpn = RPN(in_channels=512)
        self.roi_pool = ROIPoolingLayer(pool_size=pool_size, spatial_scale=1 / 8)
        self.heads = Heads(in_channels=512, pool_size=pool_size, num_classes=num_classes)

    def forward(self, images, rois):
        feature_map = self.backbone(images)
        rpn_class, rpn_bbox = self.rpn(feature_map)
        pooled_rois = self.roi_pool(feature_map, rois)
        classifier, bbox_regressor = self.heads(pooled_rois)
        return rpn_class, rpn_bbox, classifier, bbox_regressor


# Step 6: Calculate Intersection over Union (IoU)
def calculate_iou(box1, box2):
    ymin1, xmin1, ymax1, xmax1 = box1
    ymin2, xmin2, ymax2, xmax2 = box2

    inter_xmin = max(xmin1, xmin2)
    inter_ymin = max(ymin1, ymin2)
    inter_xmax = min(xmax1, xmax2)
    inter_ymax = min(ymax1, ymax2)

    inter_area = max(0, inter_xmax - inter_xmin) * max(0, inter_ymax - inter_ymin)
    area1 = (xmax1 - xmin1) * (ymax1 - ymin1)
    area2 = (xmax2 - xmin2) * (ymax2 - ymin2)
    union_area = area1 + area2 - inter_area

    iou = inter_area / union_area if union_area != 0 else 0
    return iou


# Step 7: Calculate Precision and Recall
def calculate_precision_recall(
    pred_boxes,
    pred_scores,
    pred_labels,
    gt_boxes,
    gt_labels,
    iou_threshold=0.5,
    score_threshold=0.0,
):
    tp, fp = 0, 0
    matched_gt = set()

    for i, pred_box in enumerate(pred_boxes):
        if pred_scores[i] < score_threshold:
            continue

        best_iou = 0
        best_gt_index = -1

        for j, gt_box in enumerate(gt_boxes):
            if pred_labels[i] == gt_labels[j] and j not in matched_gt:
                iou = calculate_iou(pred_box, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_index = j

        if best_iou >= iou_threshold:
            tp += 1
            matched_gt.add(best_gt_index)
        else:
            fp += 1

    fn = len(gt_boxes) - len(matched_gt)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    return precision, recall


# Step 8: Calculate Average Precision (AP)
def calculate_ap(precisions, recalls):
    precisions = [0.0] + precisions + [0.0]
    recalls = [0.0] + recalls + [1.0]

    for i in range(len(precisions) - 1, 0, -1):
        precisions[i - 1] = max(precisions[i - 1], precisions[i])

    indices = [i for i in range(1, len(recalls)) if recalls[i] != recalls[i - 1]]
    ap = sum((recalls[i] - recalls[i - 1]) * precisions[i] for i in indices)
    return ap


# Step 9: Calculate Mean Average Precision (mAP) Across All Classes
def calculate_map(predictions, ground_truths, num_classes, iou_threshold=0.5):
    average_precisions = []

    for class_id in range(1, num_classes):
        all_precisions = []
        all_recalls = []

        for pred, gt in zip(predictions, ground_truths):
            pred_boxes = [box for i, box in enumerate(pred['boxes']) if pred['labels'][i] == class_id]
            pred_scores = [score for i, score in enumerate(pred['scores']) if pred['labels'][i] == class_id]
            gt_boxes = [box for i, box in enumerate(gt['boxes']) if gt['labels'][i] == class_id]

            precision, recall = calculate_precision_recall(
                pred_boxes,
                pred_scores,
                [class_id] * len(pred_boxes),
                gt_boxes,
                [class_id] * len(gt_boxes),
                iou_threshold,
            )

            all_precisions.append(precision)
            all_recalls.append(recall)

        ap = calculate_ap(all_precisions, all_recalls)
        average_precisions.append(ap)

    mAP = float(np.mean(average_precisions))
    return mAP


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    num_classes = 80
    model = FasterRCNNToy(num_classes=num_classes).to(device)
    print(model)

    # Example forward pass with synthetic data
    images = torch.randn(1, 3, 224, 224, device=device)
    rois = torch.tensor(
        [
            [0, 10.0, 20.0, 100.0, 140.0],
            [0, 50.0, 60.0, 180.0, 200.0],
        ],
        dtype=torch.float32,
        device=device,
    )

    with torch.inference_mode():
        outputs = model(images, rois)
    print("Forward pass complete. Output tensor shapes:", [o.shape for o in outputs])

    # Placeholder predictions and ground truths for testing
    predictions = [
        {'boxes': [[0.1, 0.1, 0.4, 0.4], [0.5, 0.5, 0.8, 0.8]], 'scores': [0.9, 0.7], 'labels': [1, 2]}
    ]
    ground_truths = [
        {'boxes': [[0.12, 0.12, 0.42, 0.42], [0.5, 0.5, 0.8, 0.8]], 'labels': [1, 2]}
    ]

    mean_ap = calculate_map(predictions, ground_truths, num_classes)
    print(f"Mean Average Precision (mAP): {mean_ap:.2f}")

# Task 1: Change the number of layers

# Task 02: Take a data if you have and just randomly test it what it shows

# Task 03: Labelling 150 images through labelling tools and develop your own model, train and test on the data and show results.

In [ ]:
import argparse
import json
import random
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import torchvision
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.models.detection import (
    FasterRCNN_ResNet50_FPN_V2_Weights,
    fasterrcnn_resnet50_fpn_v2,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


@dataclass
class BackboneConfig:
    num_blocks: int = 3
    filters: Tuple[int, ...] = (64, 128, 256)
    dropout: float = 0.0


class ConfigurableBackbone(nn.Module):
    """Task 1: configurable educational backbone for layer-count experiments."""

    def __init__(self, config: BackboneConfig):
        super().__init__()
        if config.num_blocks < 1:
            raise ValueError("num_blocks must be >= 1")
        if len(config.filters) < config.num_blocks:
            raise ValueError("filters length must be >= num_blocks")

        layers: List[nn.Module] = []
        in_channels = 3
        for block_idx in range(config.num_blocks):
            out_channels = config.filters[block_idx]
            layers.extend(
                [
                    nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                    nn.ReLU(inplace=True),
                    nn.MaxPool2d(kernel_size=2, stride=2),
                ]
            )
            if config.dropout > 0:
                layers.append(nn.Dropout2d(p=config.dropout))
            in_channels = out_channels

        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class COCODetectionDataset(Dataset):
    """Minimal COCO JSON reader without pycocotools."""

    def __init__(self, images_dir: Path, annotation_file: Path):
        if not images_dir.exists() or not images_dir.is_dir():
            raise FileNotFoundError(f"images_dir does not exist: {images_dir}")
        if not annotation_file.exists() or not annotation_file.is_file():
            raise FileNotFoundError(f"annotation_file does not exist: {annotation_file}")

        with annotation_file.open("r", encoding="utf-8") as f:
            coco = json.load(f)

        images = coco.get("images", [])
        annotations = coco.get("annotations", [])
        categories = coco.get("categories", [])

        if not images or not annotations or not categories:
            raise ValueError("COCO JSON must include non-empty 'images', 'annotations', and 'categories'.")

        self.images_dir = images_dir
        self.id_to_image = {img["id"]: img for img in images}

        cat_ids = sorted(cat["id"] for cat in categories)
        self.cat_id_to_label = {cat_id: idx + 1 for idx, cat_id in enumerate(cat_ids)}
        self.label_to_name = {
            self.cat_id_to_label[cat["id"]]: cat["name"]
            for cat in categories
            if cat["id"] in self.cat_id_to_label
        }

        anns_by_image: Dict[int, List[dict]] = {}
        for ann in annotations:
            anns_by_image.setdefault(ann["image_id"], []).append(ann)

        self.samples: List[Tuple[Path, List[dict], int]] = []
        for image_id, img in self.id_to_image.items():
            file_name = img["file_name"]
            image_path = images_dir / file_name
            image_anns = anns_by_image.get(image_id, [])
            if not image_path.exists() or len(image_anns) == 0:
                continue
            self.samples.append((image_path, image_anns, image_id))

        if not self.samples:
            raise ValueError("No usable image/annotation pairs found. Check image paths and annotations.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index: int):
        image_path, anns, image_id = self.samples[index]
        image = Image.open(image_path).convert("RGB")

        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for ann in anns:
            x, y, w, h = ann["bbox"]
            xmin = float(x)
            ymin = float(y)
            xmax = float(x + w)
            ymax = float(y + h)
            if xmax <= xmin or ymax <= ymin:
                continue
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.cat_id_to_label[ann["category_id"]])
            areas.append(float(w * h))
            iscrowd.append(int(ann.get("iscrowd", 0)))

        if not boxes:
            raise ValueError(f"Image {image_path} has no valid boxes.")

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([image_id], dtype=torch.int64),
            "area": torch.tensor(areas, dtype=torch.float32),
            "iscrowd": torch.tensor(iscrowd, dtype=torch.int64),
        }

        return F.to_tensor(image), target


def collate_fn(batch):
    return tuple(zip(*batch))


def get_detection_model(num_classes: int, device: torch.device) -> nn.Module:
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn_v2(weights=weights)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    model.to(device)
    return model


def gather_images(folder: Path, recursive: bool = True) -> List[Path]:
    if recursive:
        candidates = [p for p in folder.rglob("*") if p.suffix.lower() in IMAGE_EXTS]
    else:
        candidates = [p for p in folder.glob("*") if p.suffix.lower() in IMAGE_EXTS]
    return sorted(candidates)


def run_folder_inference(output_dir: Path, score_threshold: float = 0.5):
    user_input = input("Enter image folder path for batch inference: ").strip()
    input_dir = Path(user_input).expanduser().resolve()

    if not input_dir.exists() or not input_dir.is_dir():
        raise FileNotFoundError(f"Invalid folder path: {input_dir}")

    image_paths = gather_images(input_dir, recursive=True)
    if not image_paths:
        raise ValueError(
            f"No supported images found in {input_dir}. Supported: {sorted(IMAGE_EXTS)}"
        )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn_v2(weights=weights).to(device)
    model.eval()

    preprocess = weights.transforms()
    categories = weights.meta.get("categories", [])

    output_dir.mkdir(parents=True, exist_ok=True)
    annotated_dir = output_dir / "output_predictions"
    annotated_dir.mkdir(exist_ok=True)

    records = []
    total_with_dets = 0

    with torch.inference_mode():
        for image_path in image_paths:
            image = Image.open(image_path).convert("RGB")
            width, height = image.size
            image_tensor = preprocess(image).to(device)
            prediction = model([image_tensor])[0]

            boxes = prediction["boxes"].detach().cpu()
            labels = prediction["labels"].detach().cpu()
            scores = prediction["scores"].detach().cpu()

            preds = []
            draw = ImageDraw.Draw(image)

            for box, label, score in zip(boxes, labels, scores):
                score_v = float(score.item())
                if score_v < score_threshold:
                    continue
                box_v = [float(v) for v in box.tolist()]  # pixel coords [xmin, ymin, xmax, ymax]
                label_idx = int(label.item())
                label_name = categories[label_idx] if label_idx < len(categories) else str(label_idx)

                preds.append({"label": label_name, "score": score_v, "box": box_v})
                draw.rectangle(box_v, outline="red", width=2)
                draw.text((box_v[0], box_v[1]), f"{label_name}:{score_v:.2f}", fill="red")

            if preds:
                total_with_dets += 1

            relative_image_path = image_path.relative_to(input_dir)
            annotated_image_path = annotated_dir / relative_image_path
            annotated_image_path.parent.mkdir(parents=True, exist_ok=True)
            image.save(annotated_image_path)

            records.append(
                {
                    "image": str(relative_image_path.as_posix()),
                    "width": width,
                    "height": height,
                    "predictions": preds,
                }
            )

    predictions_json = output_dir / "predictions.json"
    with predictions_json.open("w", encoding="utf-8") as f:
        json.dump(records, f, indent=2)

    summary = {
        "total_images": len(image_paths),
        "total_images_with_detections": total_with_dets,
        "threshold": score_threshold,
        "model": "fasterrcnn_resnet50_fpn_v2",
        "date_time_utc": datetime.now(timezone.utc).isoformat(),
        "coordinates": "pixel [xmin, ymin, xmax, ymax]",
    }
    with (output_dir / "summary.json").open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(f"Saved predictions to: {predictions_json}")
    print(f"Saved summary to: {output_dir / 'summary.json'}")
    print(f"Annotated images in: {annotated_dir}")


def run_training_pipeline(output_dir: Path, epochs: int = 2, batch_size: int = 2, lr: float = 5e-4):
    print("Provide dataset paths for training (COCO JSON format).")
    dataset_root = Path(input("Dataset root folder path: ").strip()).expanduser().resolve()
    images_subdir = input("Images subfolder (example: images): ").strip()
    ann_rel_path = input("Annotation JSON path relative to root (example: annotations/train.json): ").strip()

    images_dir = dataset_root / images_subdir
    annotation_file = dataset_root / ann_rel_path

    if not dataset_root.exists() or not dataset_root.is_dir():
        raise FileNotFoundError(f"Dataset root not found: {dataset_root}")

    dataset = COCODetectionDataset(images_dir=images_dir, annotation_file=annotation_file)

    n_total = len(dataset)
    if n_total < 5:
        raise ValueError("Need at least 5 labeled images for a train/val/test split.")

    n_train = int(0.7 * n_total)
    n_val = int(0.15 * n_total)
    n_test = n_total - n_train - n_val
    if n_test < 1:
        n_test = 1
        n_train = max(1, n_train - 1)

    generator = torch.Generator().manual_seed(42)
    train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=generator)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)

    num_classes = len(dataset.label_to_name) + 1  # + background
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = get_detection_model(num_classes=num_classes, device=device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    best_val_loss = float("inf")
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_total = 0.0
        for images, targets in train_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in tgt.items()} for tgt in targets]

            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss_total += float(loss.item())

        model.train()
        val_loss_total = 0.0
        with torch.no_grad():
            for images, targets in val_loader:
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in tgt.items()} for tgt in targets]
                loss_dict = model(images, targets)
                val_loss_total += float(sum(loss_dict.values()).item())

        avg_train_loss = train_loss_total / max(1, len(train_loader))
        avg_val_loss = val_loss_total / max(1, len(val_loader))
        history.append({"epoch": epoch, "train_loss": avg_train_loss, "val_loss": avg_val_loss})

        print(f"Epoch {epoch}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

    if best_state is None:
        raise RuntimeError("Training did not produce a best model state.")

    model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    output_dir.mkdir(parents=True, exist_ok=True)
    model_path = output_dir / "best_model.pt"
    torch.save(model.state_dict(), model_path)

    test_predictions = []
    with torch.inference_mode():
        for images, targets in test_loader:
            image = images[0].to(device)
            pred = model([image])[0]
            tgt = targets[0]
            image_id = int(tgt["image_id"].item())
            pred_boxes = pred["boxes"].detach().cpu().tolist()
            pred_labels = pred["labels"].detach().cpu().tolist()
            pred_scores = pred["scores"].detach().cpu().tolist()

            mapped_preds = []
            for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
                mapped_preds.append(
                    {
                        "label": dataset.label_to_name.get(int(label), str(label)),
                        "score": float(score),
                        "box": [float(v) for v in box],
                    }
                )

            test_predictions.append({"image_id": image_id, "predictions": mapped_preds})

    with (output_dir / "test_predictions.json").open("w", encoding="utf-8") as f:
        json.dump(test_predictions, f, indent=2)

    with (output_dir / "training_summary.json").open("w", encoding="utf-8") as f:
        json.dump(
            {
                "split_sizes": {"train": n_train, "val": n_val, "test": n_test},
                "history": history,
                "best_val_loss": best_val_loss,
                "model_path": str(model_path),
                "test_predictions_file": str(output_dir / "test_predictions.json"),
                "coordinates": "pixel [xmin, ymin, xmax, ymax]",
            },
            f,
            indent=2,
        )

    print(f"Training finished. Best validation loss: {best_val_loss:.4f}")
    print(f"Saved model to: {model_path}")
    print(f"Saved test predictions to: {output_dir / 'test_predictions.json'}")


def show_task1_demo(args):
    config = BackboneConfig(
        num_blocks=args.num_blocks,
        filters=tuple(args.filters),
        dropout=args.dropout,
    )
    model = ConfigurableBackbone(config)
    dummy = torch.randn(1, 3, 224, 224)
    output = model(dummy)
    print(model)
    print(f"Task-1 demo output shape: {tuple(output.shape)}")


def parse_args():
    parser = argparse.ArgumentParser(description="Assignment pipeline for Faster R-CNN custom tasks.")
    sub = parser.add_subparsers(dest="command", required=True)

    p_task1 = sub.add_parser("task1", help="Task 1: change the number of layers.")
    p_task1.add_argument("--num-blocks", type=int, default=3)
    p_task1.add_argument("--filters", type=int, nargs="+", default=[64, 128, 256, 512])
    p_task1.add_argument("--dropout", type=float, default=0.0)

    p_infer = sub.add_parser("infer-folder", help="Task 2: run inference on all images in a user-provided folder.")
    p_infer.add_argument("--output-dir", type=Path, default=Path("runs/inference"))
    p_infer.add_argument("--threshold", type=float, default=0.5)

    p_train = sub.add_parser("train", help="Task 3: train/test custom model with labeled COCO-format data.")
    p_train.add_argument("--output-dir", type=Path, default=Path("runs/training"))
    p_train.add_argument("--epochs", type=int, default=2)
    p_train.add_argument("--batch-size", type=int, default=2)
    p_train.add_argument("--lr", type=float, default=5e-4)

    return parser.parse_args()


def main():
    args = parse_args()

    if args.command == "task1":
        show_task1_demo(args)
    elif args.command == "infer-folder":
        run_folder_inference(output_dir=args.output_dir, score_threshold=args.threshold)
    elif args.command == "train":
        run_training_pipeline(
            output_dir=args.output_dir,
            epochs=args.epochs,
            batch_size=args.batch_size,
            lr=args.lr,
        )
    else:
        raise ValueError(f"Unknown command: {args.command}")


if __name__ == "__main__":
    main()